# Chapter 20: Data Engineering for Machine Learning
**Part VI — MLOps & Production AI**

*The Complete MSc Reference: Artificial Intelligence & Machine Learning*  
*Jijesh Puliyappottammal — Digichange Publication, 2026*

---

ML data pipelines, feature engineering, feature stores, data versioning, and validation.

## Learning Objectives
See Chapter 20 in the main textbook for full learning objectives, theory, and exam guidance.

## How to Use These Notebooks
Run cells from top to bottom. All cells are self-contained. Install any missing packages with `pip install <package>` in a new cell.


In [ ]:
# Standard imports used throughout the book
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

# Verify Python and key package versions
import sys
print("Python:", sys.version.split()[0])
try:
    import numpy, sklearn, torch
    print("NumPy:", numpy.__version__)
    print("Scikit-learn:", sklearn.__version__)
    print("PyTorch:", torch.__version__)
except ImportError as e:
    print(f"Missing: {e.name} — run: pip install {e.name}")


## Define expectations on a dataset


In [ ]:
import great_expectations as gx

context = gx.get_context()

# Define expectations on a dataset
validator = context.sources.pandas_default.read_csv("data.csv")

validator.expect_column_values_to_not_be_null("user_id")
validator.expect_column_values_to_be_between("age", min_value=0, max_value=120)
validator.expect_column_values_to_be_in_set("status", ["active","churned","trial"])
validator.expect_column_mean_to_be_between("revenue", min_value=50, max_value=500)

results = validator.validate()
print("Success:", results.success)

## Load model at startup


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib, numpy as np

app = FastAPI(title="ML Prediction API")

# Load model at startup
model = joblib.load("model.pkl")
scaler = joblib.load("scaler.pkl")

class PredictionRequest(BaseModel):
    features: list[float]

class PredictionResponse(BaseModel):
    prediction: int
    probability: float

@app.post("/predict", response_model=PredictionResponse)
async def predict(req: PredictionRequest):
    X = np.array(req.features).reshape(1, -1)
    X_scaled = scaler.transform(X)
    pred = int(model.predict(X_scaled)[0])
    prob = float(model.predict_proba(X_scaled)[0, pred])
    return PredictionResponse(prediction=pred, probability=prob)

@app.get("/health")
def health(): return {"status": "healthy"}

## Training-serving skew demonstration


In [ ]:
# Training-serving skew demonstration
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

np.random.seed(42)

# Simulate a dataset where "age" has different statistics in prod vs training
def generate_data(n, age_mean=35, age_std=10):
    age     = np.random.normal(age_mean, age_std, n)
    income  = age * 1500 + np.random.normal(0, 5000, n)
    target  = ((income > 50000) & (age > 30)).astype(int)
    return pd.DataFrame({'age': age, 'income': income, 'target': target})

# Training data (age ~35)
train_df = generate_data(2000, age_mean=35, age_std=10)
X_train  = train_df[['age','income']]
y_train  = train_df['target']

# Test data (same distribution) — represents held-out validation
test_df  = generate_data(500, age_mean=35, age_std=10)
X_test   = test_df[['age','income']]
y_test   = test_df['target']

# Production data (age distribution shifted — e.g. older users)
prod_df  = generate_data(500, age_mean=50, age_std=15)
X_prod   = prod_df[['age','income']]
y_prod   = prod_df['target']

# ── Case 1: Scaler fit on training data (correct) ────────────────────────
scaler = StandardScaler().fit(X_train)
clf = LogisticRegression().fit(scaler.transform(X_train), y_train)
acc_test = accuracy_score(y_test, clf.predict(scaler.transform(X_test)))
acc_prod = accuracy_score(y_prod, clf.predict(scaler.transform(X_prod)))

print("Experiment: Distribution shift (training-serving skew)")
print("=" * 55)
print(f"Training mean age:     {X_train['age'].mean():.1f}")
print(f"Production mean age:   {X_prod['age'].mean():.1f}")
print()
print(f"Accuracy on test set (same distribution):  {acc_test:.4f}")
print(f"Accuracy on production (shifted dist):     {acc_prod:.4f}")
print(f"Performance DROP due to distribution shift: {acc_test - acc_prod:.4f}")
print()
print("This is training-serving skew in action.")
print("Solutions: feature stores, monitoring, scheduled retraining.")

---

## 📚 Review Questions

See Chapter 20 of the textbook for:
- 5 review questions
- Common exam question with model answer (Appendix C)
- Flashcard summary (Appendix A)
- Formula sheet (Appendix B)

## 📖 Further Reading

See the Further Reading section at the end of Chapter 20 in the textbook.

---
*© 2026 Jijesh Puliyappottammal — Digichange Publication. Code examples are released under the MIT Licence for educational use.*